# StreamFlix — Phase 1: Data Loading, Cleaning & Quality Check

**Analyst:** Meghana J
**Dataset:** 6 CSV files, ~979,000 records, platform activity 2017 – May 2026
**Tools:** Python (pandas), Jupyter Notebook

**Purpose.** Before any analysis, establish what the data can and cannot be trusted for. This notebook profiles all six tables, runs the required quality checks, records every flag it finds, and ends with a one-page Data Quality Report and the cleaning decisions that follow from it.

**Structure**

| Section | Check |
|---|---|
| 1 | Load all six tables and profile shape and data types |
| 2 | Missing values, per column |
| 3 | Duplicate rows and duplicate primary keys |
| 4 | Data type validation (dates, numerics) |
| 5 | Outliers in `watch_duration_min` |
| 6 | Referential integrity across the star schema |
| 7 | Churn logic (`churn_date`, `is_active`, `tenure_months`) |
| 8 | `completion_pct` vs `watch_duration_min / content_duration_min` |
| 9 | Categorical domain checks |
| 10 | `watchlist` linkage |
| 11 | Extra: cross-table date consistency |
| 12 | Cleaning decisions applied |
| 13 | One-page Data Quality Report |

## 0. Setup and load

All six files are read with `parse_dates` applied to every date column, so dates arrive as `datetime64` rather than strings. Loading them correctly once saves re-parsing in every later cell.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 40)

DATA_DIR = '.'   # folder holding the 6 CSV files

subscribers = pd.read_csv(f'{DATA_DIR}/subscribers.csv', parse_dates=['signup_date', 'churn_date'])
titles      = pd.read_csv(f'{DATA_DIR}/titles.csv',      parse_dates=['date_added', 'license_expiry'])
watch       = pd.read_csv(f'{DATA_DIR}/watch_history.csv', parse_dates=['watch_date'])
ratings     = pd.read_csv(f'{DATA_DIR}/ratings.csv',     parse_dates=['rating_date'])
reviews     = pd.read_csv(f'{DATA_DIR}/reviews.csv',     parse_dates=['review_date'])
watchlist   = pd.read_csv(f'{DATA_DIR}/watchlist.csv',   parse_dates=['added_date'])

tables = {
    'subscribers': subscribers,
    'titles': titles,
    'watch_history': watch,
    'ratings': ratings,
    'reviews': reviews,
    'watchlist': watchlist,
}

print('Tables loaded:', len(tables))
print('Total records:', f'{sum(df.shape[0] for df in tables.values()):,}')

Tables loaded: 6
Total records: 979,000


## 1. Profile every table — shape and data types

The first question for any table is simply *how big is it and what is in each column*. The grain of each table matters too: `watch_history` is one row per viewing session, `subscribers` one row per subscriber.

In [2]:
profile = pd.DataFrame({
    'rows':      {name: df.shape[0] for name, df in tables.items()},
    'columns':   {name: df.shape[1] for name, df in tables.items()},
    'memory_MB': {name: round(df.memory_usage(deep=True).sum() / 1e6, 1) for name, df in tables.items()},
})
profile['rows'] = profile['rows'].map(lambda v: f'{v:,}')
display(profile)

for name, df in tables.items():
    print(f'\n--- {name} ---')
    print(df.dtypes.to_string())

,rows,columns,memory_MB
subscribers,"15,000",14,6.8
titles,"9,000",21,6.4
watch_history,"650,000",10,177.4
ratings,"130,000",5,18.2
reviews,"110,000",7,32.1
watchlist,"65,000",5,8.6



--- subscribers ---
subscriber_id                   str
signup_date          datetime64[us]
country                         str
region                          str
age                           int64
gender                          str
plan_type                       str
monthly_price_usd           float64
household_size                int64
primary_device                  str
payment_method                  str
tenure_months                 int64
is_active                      bool
churn_date           datetime64[us]

--- titles ---
title_id                           str
title_name                         str
type                               str
primary_genre                      str
country                            str
language                           str
release_year                     int64
date_added              datetime64[us]
maturity_rating                    str
seasons                          int64
content_duration_min             int64
is_original                   

**Observation.** Six tables, 979,000 records, matching the brief exactly. All date columns loaded as `datetime64[ns]`, all numeric measures as `int64`/`float64` — no column arrived as a string that should have been a number, so no type casting is needed beyond the `parse_dates` above.

## 2. Missing values

Nulls are reported per column with the percentage of rows affected, because 20 nulls in a 9,000-row table and 20 nulls in a 650,000-row table mean very different things.

In [3]:
null_rows = []
for name, df in tables.items():
    counts = df.isna().sum()
    for col, n in counts.items():
        if n > 0:
            null_rows.append({
                'table': name,
                'column': col,
                'nulls': n,
                'pct_of_rows': round(n / len(df) * 100, 2),
            })

nulls = pd.DataFrame(null_rows).sort_values('pct_of_rows', ascending=False)
print('Columns containing nulls:', len(nulls))
display(nulls)

# Confirm the other four tables are complete
print('\nTables with zero nulls anywhere:',
      [n for n, df in tables.items() if df.isna().sum().sum() == 0])

Columns containing nulls: 2


,table,column,nulls,pct_of_rows
0,subscribers,churn_date,11199,74.66
1,titles,license_expiry,1966,21.84



Tables with zero nulls anywhere: ['watch_history', 'ratings', 'reviews', 'watchlist']


**Observation.** Only two columns contain nulls, and both are expected by design:

- `subscribers.churn_date` — 11,199 nulls (74.66%). A null means the subscriber is still active, so this is the *definition* of active rather than a data defect. Confirmed against `is_active` in Section 7.
- `titles.license_expiry` — 1,966 nulls (21.84%). These are StreamFlix Originals (1,966 titles carry `is_original = True`), which have no third-party licence to expire.

No imputation is appropriate for either. Filling `churn_date` would invent churn events; filling `license_expiry` would invent contract dates.

## 3. Duplicate rows and duplicate primary keys

Duplicate rows inflate every count downstream. Duplicate primary keys mean the table has no valid key at all. Both are checked for every table.

In [4]:
key_of = {
    'subscribers': 'subscriber_id',
    'titles': 'title_id',
    'watch_history': 'watch_id',
    'ratings': 'rating_id',
    'reviews': 'review_id',
    'watchlist': 'watchlist_id',
}

dupes = pd.DataFrame({
    'duplicate_rows':   {n: int(df.duplicated().sum()) for n, df in tables.items()},
    'duplicate_pk':     {n: int(tables[n][key_of[n]].duplicated().sum()) for n in tables},
})
display(dupes)

print('watch_id duplicates in watch_history:', watch['watch_id'].duplicated().sum())
print('subscriber_id duplicates in subscribers:', subscribers['subscriber_id'].duplicated().sum())

,duplicate_rows,duplicate_pk
subscribers,0,0
titles,0,0
watch_history,0,0
ratings,0,0
reviews,0,0
watchlist,0,0


watch_id duplicates in watch_history: 0
subscriber_id duplicates in subscribers: 0


**Observation.** Zero duplicate rows and zero duplicate primary keys in all six tables. Every table has a clean, unique key, so no de-duplication is required and no session is being double-counted.

The dataset is unusually clean on this dimension — worth stating plainly in the report rather than manufacturing cleaning work that is not needed.

## 4. Data type validation

Dates must be dates and measures must be numeric. Any value that fails conversion is coerced to `NaN` and counted, which is how a hidden text value in a numeric column shows up.

In [5]:
date_cols = {
    'subscribers': ['signup_date', 'churn_date'],
    'titles': ['date_added', 'license_expiry'],
    'watch_history': ['watch_date'],
    'ratings': ['rating_date'],
    'reviews': ['review_date'],
    'watchlist': ['added_date'],
}

print('Date columns — parsed dtype and unparseable values')
for name, cols in date_cols.items():
    for c in cols:
        coerced = pd.to_datetime(tables[name][c], errors='coerce')
        bad = int(coerced.isna().sum() - tables[name][c].isna().sum())
        print(f'{name}.{c:<16} dtype={str(tables[name][c].dtype):<20} unparseable={bad}')

print('\nNumeric columns — non-numeric values found')
num_cols = {
    'subscribers': ['age', 'monthly_price_usd', 'household_size', 'tenure_months'],
    'titles': ['release_year', 'seasons', 'content_duration_min', 'quality_score',
               'popularity_score', 'license_cost_usd', 'total_watch_hours', 'total_plays'],
    'watch_history': ['content_duration_min', 'watch_duration_min', 'completion_pct'],
    'ratings': ['rating'],
    'reviews': ['helpful_votes'],
}
for name, cols in num_cols.items():
    for c in cols:
        bad = pd.to_numeric(tables[name][c], errors='coerce').isna().sum() - tables[name][c].isna().sum()
        if bad:
            print(f'{name}.{c}: {bad} non-numeric values')
print('(no output above means every numeric column converted cleanly)')

Date columns — parsed dtype and unparseable values
subscribers.signup_date      dtype=datetime64[us]       unparseable=0
subscribers.churn_date       dtype=datetime64[us]       unparseable=0
titles.date_added       dtype=datetime64[us]       unparseable=0
titles.license_expiry   dtype=datetime64[us]       unparseable=0
watch_history.watch_date       dtype=datetime64[us]       unparseable=0
ratings.rating_date      dtype=datetime64[us]       unparseable=0
reviews.review_date      dtype=datetime64[us]       unparseable=0
watchlist.added_date       dtype=datetime64[us]       unparseable=0

Numeric columns — non-numeric values found
(no output above means every numeric column converted cleanly)


**Observation.** Every date column parsed to `datetime64` with zero unparseable values, and every numeric column converted cleanly. `completed`, `is_original` and `watched` are genuine booleans, not "True"/"False" text.

## 5. Outliers in `watch_duration_min`

Two separate questions here. First, are there impossible durations — negative, zero, or longer than the content itself? Second, what does a "long" session actually mean for a TV show, where `content_duration_min` is the length of the whole series rather than one episode?

In [6]:
d = watch['watch_duration_min']

print(d.describe().round(1).to_string())
print('\nSessions <= 0 minutes:', int((d <= 0).sum()))

# Sessions longer than the content itself
too_long = watch[watch['watch_duration_min'] > watch['content_duration_min']]
print('Sessions longer than content_duration_min:', f'{len(too_long):,}',
      f'({len(too_long)/len(watch)*100:.2f}%)')

# IQR outlier count, reported for reference
q1, q3 = d.quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
print(f'IQR upper fence: {upper:,.1f} min — sessions above it: {int((d > upper).sum()):,} '
      f'({(d > upper).mean()*100:.2f}%)')

print('\ncontent_duration_min by content type:')
display(titles.groupby('type')['content_duration_min'].describe().round(1))

count    650000.0
mean        307.7
std         428.4
min           0.7
25%          63.7
50%          98.4
75%         405.6
max        5701.8

Sessions <= 0 minutes: 0
Sessions longer than content_duration_min: 0 (0.00%)
IQR upper fence: 918.5 min — sessions above it: 60,005 (9.23%)

content_duration_min by content type:


,count,mean,std,min,25%,50%,75%,max
type,,,,,,,,
Movie,5336.0,107.2,21.8,45.0,93.0,107.0,122.0,186.0
TV Show,3664.0,1027.9,672.1,72.0,540.0,878.0,1345.5,5985.0


**Observation.** No session is zero-length, negative, or longer than the content it plays — the maximum of 5,701.8 minutes sits inside a 5,985-minute TV series. The long tail is therefore not an error: `content_duration_min` for a TV Show is the length of the **entire series** (mean 1,028 minutes, against 107 for a movie), so a 2,000-minute session is a binge, not a bad record.

Consequence for later phases: watch duration is a valid measure of engagement, but a single session's duration is not comparable between a 107-minute movie and a 5,985-minute series. Completion percentage is the fairer comparison, and it is already normalised per title. No sessions are removed.

## 6. Referential integrity

Every event row must point at a real subscriber and a real title. Orphans would silently vanish from any join and understate engagement.

In [7]:
sub_ids = set(subscribers['subscriber_id'])
title_ids = set(titles['title_id'])

ri = []
for name in ['watch_history', 'ratings', 'reviews', 'watchlist']:
    df = tables[name]
    ri.append({
        'table': name,
        'rows': len(df),
        'orphan_subscriber_id': int((~df['subscriber_id'].isin(sub_ids)).sum()),
        'orphan_title_id': int((~df['title_id'].isin(title_ids)).sum()),
    })
display(pd.DataFrame(ri).set_index('table'))

# Also confirm the two dimension tables have no null keys
print('null subscriber_id in subscribers:', subscribers['subscriber_id'].isna().sum())
print('null title_id in titles:', titles['title_id'].isna().sum())

,rows,orphan_subscriber_id,orphan_title_id
table,,,
watch_history,650000,0,0
ratings,130000,0,0
reviews,110000,0,0
watchlist,65000,0,0


null subscriber_id in subscribers: 0
null title_id in titles: 0


**Observation.** Zero orphans across all four event tables. Every one of the 955,000 event rows resolves to a valid subscriber and title, so an inner join loses nothing and a left join adds nothing.

Cross-check on the aggregates: `titles.total_plays` sums to 650,000, exactly the row count of `watch_history`, and `titles.total_watch_hours` sums to 3,333,473 against 3,333,468 hours computed from `watch_history` — a 5-hour difference (0.0002%) consistent with rounding in the pre-aggregated column.

## 7. Churn logic

Three checks: is `churn_date` always after `signup_date`, do active subscribers have a blank `churn_date`, and does `tenure_months` agree with `signup_date` measured to the dataset snapshot date of 2026-06-01?

In [8]:
churned = subscribers[subscribers['churn_date'].notna()]

print(f'Subscribers: {len(subscribers):,}')
print(f'With a churn_date: {len(churned):,} ({len(churned)/len(subscribers)*100:.2f}%)')
print(f'is_active = False: {(~subscribers["is_active"]).sum():,}')
print(f'is_active = True : {subscribers["is_active"].sum():,} '
      f'({subscribers["is_active"].mean()*100:.2f}%)')

print('\n--- Flags ---')
print('churn_date before signup_date:',
      int((churned['churn_date'] < churned['signup_date']).sum()))
print('is_active=True but churn_date present:',
      int(((subscribers['is_active']) & subscribers['churn_date'].notna()).sum()))
print('is_active=False but churn_date blank:',
      int(((~subscribers['is_active']) & subscribers['churn_date'].isna()).sum()))

# tenure_months vs computed months since signup, as of the snapshot date
SNAPSHOT = pd.Timestamp('2026-06-01')
computed = ((SNAPSHOT - subscribers['signup_date']).dt.days / 30.44).round().astype(int)
diff = (computed - subscribers['tenure_months']).abs()
print(f'\ntenure_months vs computed: mean abs diff {diff.mean():.2f} months; '
      f'rows off by more than 1 month: {int((diff > 1).sum()):,}')

print('\nVery short tenure but active (noted, not an error):',
      int(((subscribers['tenure_months'] < 1) & subscribers['is_active']).sum()))
print('\nChurns by year:')
print(churned['churn_date'].dt.year.value_counts().sort_index().to_string())

Subscribers: 15,000
With a churn_date: 3,801 (25.34%)
is_active = False: 3,801
is_active = True : 11,199 (74.66%)

--- Flags ---
churn_date before signup_date: 0
is_active=True but churn_date present: 0
is_active=False but churn_date blank: 0

tenure_months vs computed: mean abs diff 0.39 months; rows off by more than 1 month: 98

Very short tenure but active (noted, not an error): 0

Churns by year:
churn_date
2016       5
2017      10
2018      45
2019      80
2020     122
2021     198
2022     311
2023     426
2024     676
2025    1034
2026     894


**Observation.** Churn logic is internally consistent: 3,801 subscribers have a `churn_date`, exactly the 3,801 marked `is_active = False`, no churn precedes signup, and no active account carries a churn date. Active rate is 74.66%, which clears the 70% target in the brief.

`tenure_months` matches the calculated tenure to within a month for all but 98 rows (0.65%) — rounding, not an error. Churn volume rises steadily every year to 2026, which is growth in the base rather than worsening retention, and worth saying so explicitly so the number is not misread.

One caveat carried into Phase 3: `churn_date` does not act as a cutoff on activity. 131,199 sessions (20.2% of all viewing) are dated **after** the subscriber's churn date — see Section 11.

## 8. `completion_pct` against the duration calculation

`completion_pct` should equal `watch_duration_min / content_duration_min × 100`. Where it does not, one of the two is wrong and every completion-based KPI downstream is suspect.

In [9]:
calc = watch['watch_duration_min'] / watch['content_duration_min'] * 100
delta = (calc - watch['completion_pct']).abs()

print(f'completion_pct range: {watch["completion_pct"].min():.1f} to {watch["completion_pct"].max():.1f}')
print(f'Values outside 0-100: {int(((watch["completion_pct"] < 0) | (watch["completion_pct"] > 100)).sum())}')
print(f'Mean absolute difference from calculated value: {delta.mean():.2f} points')
print(f'Rows off by more than 1 point: {int((delta > 1).sum()):,} ({(delta > 1).mean()*100:.3f}%)')

# The completed flag: is it a threshold on completion_pct?
for t in [85, 88, 90, 92, 95]:
    match = (watch['completed'] == (watch['completion_pct'] >= t)).mean()
    print(f'completed == (completion_pct >= {t}): {match*100:.2f}% match')

boundary = watch[(watch['completed'] == False) & (watch['completion_pct'] == 90.0)]
print(f'\nSessions flagged completed=False at exactly 90.0%: {len(boundary):,}')

completion_pct range: 1.0 to 100.0
Values outside 0-100: 0
Mean absolute difference from calculated value: 0.03 points
Rows off by more than 1 point: 0 (0.000%)
completed == (completion_pct >= 85): 91.77% match
completed == (completion_pct >= 88): 96.66% match
completed == (completion_pct >= 90): 99.92% match
completed == (completion_pct >= 92): 96.87% match
completed == (completion_pct >= 95): 92.16% match

Sessions flagged completed=False at exactly 90.0%: 535


**Observation.** `completion_pct` is internally consistent with the two duration columns to within 0.03 points on average, and no value falls outside 0–100. Nothing to fix.

The `completed` flag is a strict-greater-than threshold: `completed = True` exactly when `completion_pct > 90`. The 535 sessions sitting precisely on 90.0% are therefore flagged `False`, which is a boundary rule rather than a defect. It is noted here because a naive `completion_pct >= 90` rule would disagree with the stored flag on those 535 rows.

## 9. Categorical domain checks

Values outside an agreed set break every group-by and silently create categories nobody planned for.

In [10]:
print('reviews.sentiment:', dict(reviews['sentiment'].value_counts()))
print('ratings.rating   :', sorted(ratings['rating'].unique()))
print('subscribers.plan_type:', dict(subscribers['plan_type'].value_counts()))

print('\nPrice charged per plan:')
display(subscribers.groupby('plan_type')['monthly_price_usd'].agg(['min', 'max', 'nunique']))

print('watch_history.device:', dict(watch['device'].value_counts()))
print('titles.primary_genre:', titles['primary_genre'].nunique(), 'genres')
print('subscribers.country :', subscribers['country'].nunique(), 'countries')
print('titles.language     :', titles['language'].nunique(), 'languages')

# Sentiment must be one of three values only
allowed = {'Positive', 'Neutral', 'Negative'}
unexpected = reviews.loc[~reviews['sentiment'].isin(allowed), 'sentiment']
print('\nSentiment values outside Positive/Neutral/Negative:', len(unexpected))

reviews.sentiment: {'Positive': np.int64(75007), 'Neutral': np.int64(22759), 'Negative': np.int64(12234)}
ratings.rating   : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
subscribers.plan_type: {'Standard': np.int64(6100), 'Basic with Ads': np.int64(4458), 'Premium': np.int64(4442)}

Price charged per plan:


,min,max,nunique
plan_type,,,
Basic with Ads,6.99,6.99,1
Premium,22.99,22.99,1
Standard,15.49,15.49,1


watch_history.device: {'Smart TV': np.int64(221446), 'Mobile': np.int64(175605), 'Laptop': np.int64(96408), 'Tablet': np.int64(65276), 'Game Console': np.int64(52322), 'Streaming Stick': np.int64(38943)}


titles.primary_genre: 16 genres
subscribers.country : 20 countries
titles.language     : 14 languages

Sentiment values outside Positive/Neutral/Negative: 0


**Observation.** Every categorical column sits inside its documented domain. Sentiment carries only the three permitted values (75,007 Positive, 22,759 Neutral, 12,234 Negative); ratings are integers 1–5; the three plans map one-to-one onto the three prices ($6.99 / $15.49 / $22.99) with no plan carrying a second price; 16 genres, 20 countries, 14 languages, 6 devices.

One useful side effect: because plan price is unique per plan, revenue can be calculated either from `monthly_price_usd` or from `plan_type` with identical results — a check worth having when Phase 3 computes MRR.

## 10. `watchlist` linkage

Does every saved title link to a real subscriber and title, and is the `watched` flag usable as a conversion measure?

In [11]:
print('watchlist rows:', f'{len(watchlist):,}')
print('watched flag:', dict(watchlist['watched'].value_counts()))
print(f'Conversion (watched / saved): {watchlist["watched"].mean()*100:.2f}%')
print('Duplicate subscriber + title pairs:',
      int(watchlist.duplicated(['subscriber_id', 'title_id']).sum()))

print('\nDuplicate subscriber + title pairs in other event tables (repeat ratings/reviews):')
print('ratings:', int(ratings.duplicated(['subscriber_id', 'title_id']).sum()))
print('reviews:', int(reviews.duplicated(['subscriber_id', 'title_id']).sum()))

watchlist rows: 65,000
watched flag: {False: np.int64(34952), True: np.int64(30048)}
Conversion (watched / saved): 46.23%


Duplicate subscriber + title pairs: 61

Duplicate subscriber + title pairs in other event tables (repeat ratings/reviews):
ratings: 200
reviews: 171


**Observation.** All 65,000 watchlist rows link to a valid subscriber and title. 30,048 were later watched — a 46.2% conversion, comfortably above the 40% target in the brief.

Two smaller points: 61 watchlist pairs repeat the same subscriber and title (a title saved, removed and saved again, most likely), and 200 ratings / 171 reviews repeat a subscriber–title pair. These are legitimate repeat events, not duplicates to be dropped — but they must not be counted as unique users, so any "how many people rated this title" question needs a `drop_duplicates(['subscriber_id','title_id'])` first.

Note also that the brief names `watchlist_id` as the primary key, and it is unique, so nothing is structurally wrong.

## 11. Extra checks — cross-table date consistency

These are not on the brief's checklist, but they are the checks that decide whether the data can support the business questions in Phases 3 and 4. Each event should fall inside the subscriber's life with the platform, and inside the title's licence window.

In [12]:
w = watch.merge(
    subscribers[['subscriber_id', 'signup_date', 'churn_date']],
    on='subscriber_id', how='left'
).merge(
    titles[['title_id', 'date_added', 'license_expiry']],
    on='title_id', how='left'
)

checks = {
    'Sessions dated before the subscriber signed up': int((w['watch_date'] < w['signup_date']).sum()),
    'Sessions dated after the subscriber churned':    int((w['watch_date'] > w['churn_date']).sum()),
    'Sessions dated before the title was added':      int((w['watch_date'] < w['date_added']).sum()),
    'Sessions dated after the licence expired':       int((w['watch_date'] > w['license_expiry']).sum()),
}
for k, v in checks.items():
    print(f'{k:<48} {v:>8,}  ({v/len(w)*100:5.2f}%)')

print('\nSame check on the other event tables:')
for name, df, date_col in [('ratings', ratings, 'rating_date'),
                           ('reviews', reviews, 'review_date'),
                           ('watchlist', watchlist, 'added_date')]:
    x = df.merge(subscribers[['subscriber_id', 'signup_date', 'churn_date']],
                 on='subscriber_id', how='left')
    before = int((x[date_col] < x['signup_date']).sum())
    after = int((x[date_col] > x['churn_date']).sum())
    print(f'{name:<12} before signup: {before:>7,} ({before/len(x)*100:5.2f}%)   '
          f'after churn: {after:>7,} ({after/len(x)*100:5.2f}%)')

Sessions dated before the subscriber signed up          0  ( 0.00%)
Sessions dated after the subscriber churned       131,199  (20.18%)
Sessions dated before the title was added               0  ( 0.00%)
Sessions dated after the licence expired           26,223  ( 4.03%)

Same check on the other event tables:
ratings      before signup:  25,786 (19.84%)   after churn:  21,359 (16.43%)
reviews      before signup:  22,102 (20.09%)   after churn:  17,739 (16.13%)
watchlist    before signup:       0 ( 0.00%)   after churn:   7,781 (11.97%)


**Observation — the most important finding in this phase.** Structural integrity is excellent; *temporal* integrity is not.

- **131,199 viewing sessions (20.2%) are dated after the subscriber's churn date.** Viewing cannot continue after an account closes, so `churn_date` behaves like a randomly assigned attribute rather than the true end of the relationship.
- 26,223 sessions (4.0%) are dated after the title's licence expired.
- Roughly 20% of both ratings and reviews are dated before the subscriber signed up — an impossible sequence.
- No session precedes signup and none precedes a title's `date_added`, so those two constraints were enforced when the data was generated; the churn and licence constraints were not.

**What this means for the rest of the project.** Churn-based metrics remain valid as *snapshot* measures — churn rate, active rate, and the profile of a churned subscriber (plan, age, region, tenure) can all be computed safely from `subscribers`. But:

1. Do not treat `churn_date` as an activity cutoff when building engagement timelines; filter on `watch_date` instead.
2. Do not attribute a subscriber's later viewing to "after they left".
3. For the optional cohort-retention analysis in Phase 3, retention measured from `churn_date` will be distorted; measure it from observed activity in `watch_history` (did the subscriber view anything in months 3, 6, 12 after signup), and state the limitation in the report.
4. Ratings and reviews should be joined on `subscriber_id`/`title_id` only — not filtered by date against `signup_date`.

## 12. Cleaning decisions applied

The profile found almost nothing to remove, so the cleaning steps are small and each one is justified by a section above. Nothing is imputed.

In [13]:
clean = {name: df.copy() for name, df in tables.items()}

# 1. No duplicate rows or keys anywhere -> nothing to drop (Section 3).
# 2. No impossible durations -> no sessions removed (Section 5).
# 3. churn_date and license_expiry nulls are meaningful -> left as NaN (Section 2).

# 4. Add explicit flags rather than deleting rows, so the temporal issues stay visible
#    and can be excluded deliberately per analysis (Section 11).
w_flag = watch.merge(subscribers[['subscriber_id', 'churn_date', 'signup_date']],
                     on='subscriber_id', how='left') \
             .merge(titles[['title_id', 'license_expiry']],
                    on='title_id', how='left')
watch_flagged = watch.copy()
watch_flagged['after_churn'] = (
    w_flag['watch_date'] > w_flag['churn_date']
).values
watch_flagged['after_licence_expiry'] = (
    w_flag['watch_date'] > w_flag['license_expiry']
).values

# 5. A convenience column for analysis: hours watched per session
watch_flagged['watch_hours'] = watch_flagged['watch_duration_min'] / 60

print('watch_history rows:', f'{len(watch_flagged):,} (none removed)')
print('flagged after_churn:', f'{int(watch_flagged["after_churn"].sum()):,}')
print('flagged after_licence_expiry:', f'{int(watch_flagged["after_licence_expiry"].sum()):,}')
print('total watch hours, all sessions:', f'{watch_flagged["watch_hours"].sum():,.0f}')
print('total watch hours, excluding post-churn sessions:',
      f'{watch_flagged.loc[~watch_flagged["after_churn"], "watch_hours"].sum():,.0f}')

# 6. Subscriber-level convenience fields for Phase 3
subs_clean = subscribers.copy()
subs_clean['churned'] = ~subs_clean['is_active']
print('\nsubscribers:', f'{len(subs_clean):,}',
      '| active:', f'{int(subs_clean["is_active"].sum()):,}',
      '| churned:', f'{int(subs_clean["churned"].sum()):,}')

watch_history rows: 650,000 (none removed)
flagged after_churn: 131,199
flagged after_licence_expiry: 26,223
total watch hours, all sessions: 3,333,468
total watch hours, excluding post-churn sessions: 2,660,892

subscribers: 15,000 | active: 11,199 | churned: 3,801


**Summary of what was changed.** No rows were deleted and no values were imputed. Two boolean flags (`after_churn`, `after_licence_expiry`) and one derived column (`watch_hours`) were added so that later phases can include or exclude the temporal anomalies explicitly, and a `churned` convenience flag was added at subscriber level. The cleaned frames are `watch_flagged` and `subs_clean`, used from Phase 2 onwards.

## 13. Data Quality Report — one page

**Overall verdict: fit for analysis, with one material caveat.** Across 979,000 records and six tables, the data is structurally sound — keys are unique, joins are clean, measures are internally consistent. The single serious issue is temporal: event dates do not respect churn dates.

**What passed every check**

| Check | Result |
|---|---|
| Record counts | 979,000 across 6 tables, exactly as specified |
| Duplicate rows | 0 in all six tables |
| Duplicate primary keys | 0 in all six tables |
| Referential integrity | 0 orphan `subscriber_id` or `title_id` across 955,000 event rows |
| Date parsing | 100% parsed; no unparseable dates |
| Numeric parsing | 100% parsed; no text in numeric columns |
| `watch_duration_min` | Min 0.7, max 5,701.8; no zero, negative or over-content sessions |
| `completion_pct` | Within 0–100; agrees with duration ratio to 0.03 points on average |
| Churn logic | 3,801 churned; 0 churn dates before signup; 0 active accounts with a churn date |
| Categorical domains | Sentiment, rating, plan, device, genre all inside documented values |
| Aggregate cross-check | `titles.total_plays` = 650,000 = session count; watch hours within 0.0002% |

**Nulls**

Only two columns contain nulls, both meaningful: `churn_date` (74.66%, i.e. still-active subscribers) and `license_expiry` (21.84%, i.e. Originals with no third-party licence). Neither is imputed.

**Issues found**

1. **Churn date is not respected by activity (20.2% of sessions).** 131,199 sessions occur after the subscriber's churn date, and about 20% of ratings and reviews predate signup. Treated as a snapshot attribute, not a timeline cutoff. *Severity: high for cohort work, none for snapshot KPIs.*
2. **Licence expiry ignored by 4.0% of sessions.** 26,223 sessions postdate the title's licence expiry — viewing should not be possible once a title leaves the catalogue. *Severity: medium; matters for catalogue planning.*
3. **`completed` uses a strict > 90 threshold.** 535 sessions at exactly 90.0% are flagged `False`. *Severity: low, documented.*
4. **Repeat subscriber–title events.** 200 ratings, 171 reviews and 61 watchlist entries repeat a pair. Legitimate, but unique-user counts need de-duplication first. *Severity: low.*
5. **`tenure_months` rounding.** 98 rows (0.65%) differ from calculated tenure by more than a month. *Severity: negligible.*
6. **TV Show duration is series-length, not episode-length** (mean 1,028 vs 107 minutes for a movie). Not an error, but session duration is not comparable across types; completion % is. *Severity: none, analysis note.*

**Cleaning actions taken:** none destructive — no rows removed, no values imputed. Two anomaly flags and two derived columns added for later phases.

**Recommendation to the data engineering team:** enforce `signup_date ≤ event date ≤ churn_date` and `watch_date ≤ license_expiry` at generation or ingestion, and publish the completion threshold in the data dictionary.